In [1]:
import sys, os
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2-nf')
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2-nf/tdmpc/src')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['MUJOCO_GL'] = 'egl'

In [2]:
import torch
import numpy as np
import time
from pathlib import Path
from omegaconf import OmegaConf
from cfg import parse_cfg
from env import make_env
from algorithm.helper import Episode, ReplayBuffer, linear_schedule
from o2.tdmpc_o2 import TDMPC_O2
from o2.training_utils import set_seed, update_tdmpc, update_decoder
from o2.eval_utils import evaluate_agent

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
from scripts.train_o2 import make_cfg

cfg = make_cfg(
    'cheetah-run', seed=10,
    exp_name='o2_ddpg',
    train_steps=10000,
    seed_steps=1000,             # CHANGE THIS
    std_schedule='linear(0.5,0.05,2500)',
    horizon_schedule='linear(1,5,2500)',
    cem_warmstart=True,
    latent_num_samples=32,
    latent_num_elites=8,
    eval_episodes=0,
    told_updates=250,
    iterations=5,
    horizon=5,
    dcem_iterations=5,
    #load_model='models/cheetah-run20k/model.pt',
    #load_buffer='models/cheetah-run20k/replay_buffer.pth',
    diversity_coeff=0.5,
    log_det_target=-30,
    decoder_updates=80,
    num_pi_tra
# Override anything here:
# cfg.lr = 3e-4

print(OmegaConf.to_yaml(cfg))

task: cheetah-run
modality: state
action_repeat: 4
discount: 0.99
episode_length: 250
train_steps: 10000
iterations: 5
num_samples: 512
num_elites: 64
mixture_coef: 0.05
min_std: 0.05
temperature: 0.5
momentum: 0.1
batch_size: 512
max_buffer_size: 1000000
horizon: 5
reward_coef: 0.5
value_coef: 0.1
consistency_coef: 2
rho: 0.5
kappa: 0.1
lr: 0.001
std_schedule: linear(0.5,0.05,2500)
horizon_schedule: linear(1,5,2500)
per_alpha: 0.6
per_beta: 0.4
grad_clip_norm: 10
seed_steps: 1000
update_freq: 2
tau: 0.01
enc_dim: 256
mlp_dim: 512
latent_dim: 50
use_wandb: true
wandb_project: TDMPC_O2
wandb_entity: odysseaskon-national-technical-university-of-athens
seed: 10
exp_name: o2_ddpg
eval_freq: 20000
eval_episodes: 0
save_video: false
save_model: false
cem_warmstart: true
latent_num_samples: 32
latent_num_elites: 8
told_updates: 250
dcem_iterations: 5
diversity_coeff: 0.5
log_det_target: -30
decoder_updates: 80
task_title: Cheetah Run
device: cuda
use_latent_state: true
flow_num_layers: 4
flow

In [19]:
set_seed(cfg.seed)
env    = make_env(cfg)
cfg.latent_action_dim = cfg.horizon * cfg.action_dim
agent  = TDMPC_O2(cfg)
buffer = ReplayBuffer(cfg)
step, episode_idx = 0, 0
start_time = time.time()
print('Ready')

Ready


In [ ]:
if cfg.get('load_model', None):
    d = torch.load(cfg.load_model)
    state_dict = d['model'] if 'model' in d else d
    for k in [k for k in state_dict if k.startswith('_action_decoder') or k.startswith('_V')]:
        del state_dict[k]
    agent.model.load_state_dict(state_dict, strict=False)
    if 'model_target' in d:
        target_dict = d['model_target']
        for k in [k for k in target_dict if k.startswith('_action_decoder') or k.startswith('_V')]:
            del target_dict[k]
        agent.model_target.load_state_dict(target_dict, strict=False)
        print("Target loaded")
    else:
        agent.model_target.load_state_dict(agent.model.state_dict(), strict=False)
        print("Target copied")
    print(f'Loaded model from {cfg.load_model}')
if cfg.get('load_buffer', None):
    buffer.__dict__.update(torch.load(cfg.load_buffer, weights_only=False))
    print(f'Loaded buffer from {cfg.load_buffer}')


In [ ]:
num_episodes = 40
W = 38

print("Number of episodes:", num_episodes)
for i in range(num_episodes):
    obs = env.reset()
    episode = Episode(cfg, obs)
    t_ep = time.time()

    u_mean_norms, u_std_means = [], []
    while not episode.done:
        if step < cfg.seed_steps:
            action = torch.tensor(env.action_space.sample(), dtype=torch.float32, device=agent.device)
        else:
            action, u_mean, u_std, *_ = agent.CEM_in_latent(obs, step=step, sample_final_action=True, t0=episode.first)
            u_mean_norms.append(u_mean.norm().item())
            u_std_means.append(u_std.mean().item())

        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    buffer += episode
    ep_time = time.time() - t_ep

    step += cfg.episode_length
    episode_idx += 1
    env_step = int(step * cfg.action_repeat)
    print('─' * W)
    print(f'  Episode {episode_idx}   step {env_step:,}')
    print('─' * W)
    print(f'  {"Reward":<16}: {episode.cumulative_reward:>8.1f}')
    print(f'  {"Horizon":<16}: {int(linear_schedule(cfg.horizon_schedule, step)):>8}')
    print(f'  {"Std":<16}: {linear_schedule(cfg.std_schedule, step):>8.3f}')
    print(f'  {"Ep time":<16}: {ep_time:>7.1f}s')
    if step > 20000:  #cfg.seed_steps:
        print(f'  {"CEM u_mean norm":<16}: {sum(u_mean_norms)/len(u_mean_norms):>8.4f}')
        print(f'  {"CEM u_std mean":<16}: {sum(u_std_means)/len(u_std_means):>8.4f}')

    train_metrics, update_time = {}, 0.0

    if step >= cfg.seed_steps:
        t = time.time()
        train_metrics = update_tdmpc(agent, buffer, step)
        update_time = time.time() - t

    dec_metrics = {}
    decoder_time = 0.0
    
    if step >= cfg.seed_steps and i % 1 == 0:
        t = time.time()
        dec_metrics = update_decoder(agent, buffer, cfg, step)
        decoder_time = time.time() - t
        for iteration, norm in sorted(dec_metrics['grad_tracker']):
            print(f'  DCEM iter {iteration} grad norm: {norm:.6f}')

    print(f'  {"Update time":<16}: {update_time:>7.1f}s')
    
    if dec_metrics:
        print(f'  {"Decoder time":<16}: {decoder_time:>7.1f}s')
        for k, v in dec_metrics.items():
            if k != 'grad_tracker':
                print(f'  {k:<20}: {v:>8.4f}')
                
    if i % 10 == 0 and step >= cfg.seed_steps:
        eval_metrics = evaluate_agent(
            env,
            agent,
            cfg,
            step=step,
            n_episodes=1,
            policy='cem_latent',
            save_dir='eval_videos/temp7',
            video_mode='first',   # or 'first'
            sample_final_action=True,
        )
    print(f'  {"Total time":<16}: {time.time() - start_time:>7.0f}s')

    if train_metrics:
        for k, v in train_metrics.items():
            print(f'  {k:<20}: {v:>8.4f}')

In [ ]:
# Fully offline decoder training: repeatedly samples batches from the
# already-populated buffer and updates the O2 decoder (DCEM + update_decoder_stoch).
# No env stepping, no TOLD updates — run the data-collection cell above first.
num_decoder_iters = 10
log_every = 1

loss_history = []
for i in range(num_decoder_iters):
    eval_metrics = evaluate_agent(
        env,
        agent,
        cfg,
        step=step,
        n_episodes=5,
        policy='cem_latent',
        save_dir='eval_videos/temp7',
        video_mode='none',   # or 'first'
        sample_final_action=True,
    )
    dec_metrics = update_decoder(agent, buffer, cfg, step)
    loss_history.append(dec_metrics['decoder_loss'])
    if i % log_every == 0 or i == num_decoder_iters - 1:
        print(f'[{i:>4}/{num_decoder_iters}] '
              f'loss={dec_metrics["decoder_loss"]:.4f}  '
              f'value={dec_metrics["value_mean"]:.4f}  '
              f'alpha={dec_metrics["diversity_alpha"]:.4f}  '
              f'log_prob={dec_metrics["log_prob_action"]:.4f}  '
              f'grad_norm={dec_metrics["decoder_grad_norm_max"]:.4f}  '
              f'sat={dec_metrics["saturation"]:.4f}')
eval_metrics = evaluate_agent(
        env,
        agent,
        cfg,
        step=step,
        n_episodes=5,
        policy='cem_latent',
        save_dir='eval_videos/temp7',
        video_mode='none',   # or 'first'
        sample_final_action=True,
    )
